# 动态行为层可视化 - 2026-04-21

重新实现：
1. OD流线图：使用 transbigdata.visualization_od() + 导出PNG
2. 社区发现：严格按参照方法（网格化→igraph Louvain→合并多边形）+ 保存中间CSV + 导出PNG
3. 底图：style=6（浅色中文版），bounds缩小，本地缓存

In [ ]:
import sys
from pathlib import Path

project_root = Path(r"E:\00_Commute_Scenario_Research")
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import geopandas as gpd
import transbigdata as tbd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

from src import (
    OD_FEATURE_CSV, SHP_PATH, get_result_path, logger,
)
from src.data_prep import load_fence

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

print("模块加载完成")

In [ ]:
# 配置
MAPBOX_TOKEN = os.getenv("MAPBOX_TOKEN")\nif not MAPBOX_TOKEN:\n    raise RuntimeError("MAPBOX_TOKEN is not set")
BOUNDS = [112.703, 27.8, 113.284, 28.5]  # 缩小范围
OUTPUT_SECTION = '3.Situation_Diagnosis/3.1Holistic_Diagnosis/3.1.2Dynamic_Stats'
TILE_CACHE_DIR = project_root / 'tempt' / 'tileimg'

# 设置mapbox token和底图缓存路径（全局生效）
tbd.set_mapboxtoken(MAPBOX_TOKEN)
TILE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
tbd.set_imgsavepath(str(TILE_CACHE_DIR))

# 加载数据
fence = load_fence(SHP_PATH)
print(f"TAZ数: {len(fence)}")

df_od_full = pd.read_csv(OD_FEATURE_CSV, encoding='utf-8-sig')
df_od_full = df_od_full.rename(columns={'Htaz': 'o', 'Jtaz': 'd'})
print(f"OD数据行数: {len(df_od_full)}")
print(f"总人数: {df_od_full['人数'].sum():.0f}")

## 1. OD流线图（transbigdata.visualization_od）

In [ ]:
# 准备OD数据：需要 slon, slat, elon, elat, count 列
fence_pts = fence[['taz', 'center_x', 'center_y']].copy()
fence_pts['taz'] = fence_pts['taz'].astype(int)

df_od = df_od_full[['o', 'd', '人数']].copy()
df_od['o'] = df_od['o'].astype(int)
df_od['d'] = df_od['d'].astype(int)

# 合并起点坐标
df_od = df_od.merge(
    fence_pts.rename(columns={'taz': 'o', 'center_x': 'slon', 'center_y': 'slat'}),
    on='o', how='inner'
)
# 合并终点坐标
df_od = df_od.merge(
    fence_pts.rename(columns={'taz': 'd', 'center_x': 'elon', 'center_y': 'elat'}),
    on='d', how='inner'
)

# 重命名人数列为count
df_od = df_od.rename(columns={'人数': 'count'})

print(f"OD流线数据准备完成: {len(df_od)} 条")
print(f"列名: {df_od.columns.tolist()}")
print(f"总流量: {df_od['count'].sum():.0f}")
print(f"\n流量分布:")
print(df_od['count'].describe())

In [ ]:
# 使用 transbigdata.visualization_od 生成交互式地图
# 注意：需要安装 keplergl（pip install keplergl）
# 若未安装，自动降级为 plotly 版本

print("生成OD流线图...")

html_path = get_result_path(OUTPUT_SECTION, 'star_OD流线图_实际格局.html')

try:
    vmap = tbd.visualization_od(
        df_od,
        col=['slon', 'slat', 'elon', 'elat', 'count'],
        zoom='auto',
        height=800,
        accuracy=500,
        mincount=0
    )
    vmap.save_to_html(str(html_path))
    print(f"OD流线图HTML已保存（keplergl）: {html_path}")
    vmap

except ImportError:
    print("keplergl未安装，使用plotly生成OD流线图...")
    import plotly.graph_objects as go

    df_top = df_od.nlargest(2000, 'count').copy()
    val_min = df_top['count'].min()
    val_max = df_top['count'].max()
    if val_max == val_min:
        val_max = val_min + 1

    fig = go.Figure()
    for _, row in df_top.iterrows():
        norm = (row['count'] - val_min) / (val_max - val_min)
        width = 1.0 + norm * 6.0
        alpha = 0.2 + norm * 0.7
        fig.add_trace(go.Scattermapbox(
            lon=[row['slon'], row['elon']],
            lat=[row['slat'], row['elat']],
            mode='lines',
            line=dict(width=width, color=f'rgba(46,125,154,{alpha:.2f})'),
            showlegend=False,
            hoverinfo='skip',
        ))

    center_lon = (BOUNDS[0] + BOUNDS[2]) / 2
    center_lat = (BOUNDS[1] + BOUNDS[3]) / 2
    fig.update_layout(
        mapbox=dict(
            accesstoken=MAPBOX_TOKEN,
            style='light',
            center=dict(lon=center_lon, lat=center_lat),
            zoom=9,
        ),
        margin=dict(l=0, r=0, t=0, b=0),
        width=1920, height=1080,
    )
    fig.write_html(str(html_path))
    print(f"OD流线图HTML已保存（plotly）: {html_path}")

In [ ]:
# 检查为什么很多OD流未被可视化
print("\n=== OD流稀疏性诊断 ===")
print(f"总OD对数: {len(df_od)}")
print(f"\n流量分段统计:")
bins = [0, 1, 5, 10, 50, 100, 500, 1000, df_od['count'].max()]
labels = ['<1', '1-5', '5-10', '10-50', '50-100', '100-500', '500-1000', '>1000']
df_od['flow_bin'] = pd.cut(df_od['count'], bins=bins, labels=labels, include_lowest=True)
print(df_od['flow_bin'].value_counts().sort_index())

print(f"\n流量>10的OD对数: {(df_od['count'] > 10).sum()}")
print(f"流量>50的OD对数: {(df_od['count'] > 50).sum()}")
print(f"流量>100的OD对数: {(df_od['count'] > 100).sum()}")

# 可能原因：mincount=0会显示所有流，但小流量的线条可能太细看不见
# 建议：调整 mincount 参数过滤小流量

In [ ]:
# 导出为PNG（使用keplergl的截图功能或手动截图）
# 由于keplergl不直接支持PNG导出，这里提供两种方案：

# 方案1：使用playwright自动截图（需要安装playwright）
try:
    from playwright.sync_api import sync_playwright
    
    png_path = get_result_path(OUTPUT_SECTION, 'star_OD流线图_实际格局.png')
    
    with sync_playwright() as p:
        browser = p.chromium.launch()
        page = browser.new_page(viewport={'width': 1920, 'height': 1080})
        page.goto(f'file:///{html_path.as_posix()}')
        page.wait_for_timeout(5000)  # 等待地图加载
        page.screenshot(path=str(png_path), full_page=False)
        browser.close()
    
    print(f"PNG已保存: {png_path}")
except ImportError:
    print("playwright未安装，请手动截图或运行: pip install playwright && playwright install chromium")
except Exception as e:
    print(f"截图失败: {e}")
    print("请手动打开HTML文件并截图保存")

## 2. 社区发现（严格按参照方法）

In [ ]:
# Step 1: 数据网格化（参照Example 8）
print("=== Step 1: 数据网格化 ===")

# 获取网格参数（使用area_to_params）
params = tbd.area_to_params(location=BOUNDS, accuracy=500, method='rect')
print(f"网格参数: {params}")

# 聚合OD到网格
od_gdf = tbd.odagg_grid(
    df_od,
    params,
    col=['slon', 'slat', 'elon', 'elat']
)

print(f"网格化后OD对数: {len(od_gdf)}")
print(f"总流量: {od_gdf['count'].sum():.0f}")

# 保存网格化OD
csv_grid_od = get_result_path(OUTPUT_SECTION, '社区发现_01_网格化OD.csv')
od_gdf[['SLONCOL', 'SLATCOL', 'ELONCOL', 'ELATCOL', 'count']].to_csv(
    csv_grid_od, index=False, encoding='utf-8-sig'
)
print(f"网格化OD已保存: {csv_grid_od}")

In [ ]:
# Step 2: 提取节点和边（参照Example 8）
print("\n=== Step 2: 提取节点和边 ===")

# 合并LONCOL和LATCOL为节点ID
od_gdf['S'] = od_gdf['SLONCOL'].astype(str) + ',' + od_gdf['SLATCOL'].astype(str)
od_gdf['E'] = od_gdf['ELONCOL'].astype(str) + ',' + od_gdf['ELATCOL'].astype(str)

# 提取节点集
node = set(od_gdf['S']) | set(od_gdf['E'])
node = pd.DataFrame(node, columns=['grid'])
node['node_id'] = range(len(node))

print(f"节点数: {len(node)}")

# 保存节点
csv_nodes = get_result_path(OUTPUT_SECTION, '社区发现_02_节点.csv')
node.to_csv(csv_nodes, index=False, encoding='utf-8-sig')
print(f"节点已保存: {csv_nodes}")

# 提取边
node_s = node.copy()
node_s.columns = ['S', 'S_id']
od_gdf = pd.merge(od_gdf, node_s, on=['S'])

node_e = node.copy()
node_e.columns = ['E', 'E_id']
od_gdf = pd.merge(od_gdf, node_e, on=['E'])

edge = od_gdf[['S_id', 'E_id', 'count']]
print(f"边数: {len(edge)}")

# 保存边
csv_edges = get_result_path(OUTPUT_SECTION, '社区发现_03_边.csv')
edge.to_csv(csv_edges, index=False, encoding='utf-8-sig')
print(f"边已保存: {csv_edges}")

In [ ]:
# Step 3: 构建网络并进行社区发现（参照Example 8）
print("\n=== Step 3: 社区发现（igraph Louvain） ===")

import igraph as ig

# 创建网络
g = ig.Graph()
g.add_vertices(len(node))
g.add_edges(edge[['S_id', 'E_id']].values)

# 添加权重
edge_weights = edge['count'].values
for i in range(len(edge_weights)):
    g.es[i]['weight'] = edge_weights[i]

# 社区发现
g_clustered = g.community_multilevel(weights=edge_weights, return_levels=False)

# 模块度
modularity = g_clustered.modularity
print(f"模块度: {modularity:.6f}")

# 分配社区标签
node['group'] = g_clustered.membership
node.columns = ['grid', 'node_id', 'group']

print(f"社区数: {node['group'].nunique()}")
print(f"\n社区规模分布:")
print(node['group'].value_counts().head(20))

# 保存社区分配结果
csv_community = get_result_path(OUTPUT_SECTION, '社区发现_04_社区分配.csv')
node.to_csv(csv_community, index=False, encoding='utf-8-sig')
print(f"\n社区分配已保存: {csv_community}")

In [ ]:
# Step 4: 筛选大社区并生成几何（参照Example 8）
print("\n=== Step 4: 筛选大社区并生成几何 ===")

# 统计每个社区的网格数
group_counts = node['group'].value_counts()
# 保留网格数>10的社区
large_groups = group_counts[group_counts > 10].index
node_filtered = node[node['group'].isin(large_groups)].copy()

print(f"大社区数（>10网格）: {len(large_groups)}")
print(f"保留网格数: {len(node_filtered)}")

# 解析网格ID为LONCOL和LATCOL
node_filtered['LONCOL'] = node_filtered['grid'].apply(lambda r: int(r.split(',')[0]))
node_filtered['LATCOL'] = node_filtered['grid'].apply(lambda r: int(r.split(',')[1]))

# 生成网格多边形（grid_to_polygon返回列表，取第一个元素）
node_filtered['geometry'] = node_filtered.apply(
    lambda row: tbd.grid_to_polygon([row['LONCOL'], row['LATCOL']], params)[0],
    axis=1
)

# 转为GeoDataFrame
node_gdf = gpd.GeoDataFrame(node_filtered, crs='EPSG:4326')

print(f"\nGeoDataFrame shape: {node_gdf.shape}")
print(node_gdf.head(3))

In [ ]:
# Step 5: 按社区合并多边形（参照Example 8）
print("\n=== Step 5: 按社区合并多边形 ===")

# 使用transbigdata的merge_polygon
node_community = tbd.merge_polygon(node_gdf, 'group')

# 提取外边界
node_community = tbd.polyon_exterior(node_community, minarea=0.0001)

print(f"合并后社区数: {len(node_community)}")
print(node_community.head(3))

# 保存最终社区多边形
csv_final = get_result_path(OUTPUT_SECTION, 'star_社区发现_最终结果.csv')
node_community_export = node_community.copy()
node_community_export['geometry_wkt'] = node_community_export['geometry'].apply(lambda g: g.wkt)
node_community_export[['group', 'geometry_wkt']].to_csv(
    csv_final, index=False, encoding='utf-8-sig'
)
print(f"最终社区结果已保存: {csv_final}")

In [ ]:
# Step 6: 可视化（参照Example 8，使用matplotlib + 本地底图缓存）
print("\n=== Step 6: 可视化社区发现结果 ===")

# 生成调色板
n_communities = len(node_community)
cmap = sns.hls_palette(n_colors=n_communities, l=0.7, s=0.8)

# 创建图形
fig = plt.figure(figsize=(14, 14), dpi=300)
ax = plt.subplot(111)
plt.sca(ax)

# 加载底图（transbigdata自动缓存到TILE_CACHE_DIR）
tbd.plot_map(plt, bounds=BOUNDS, zoom=11, style=6)  # style=6: light-ch（浅色中文版）

# 打乱社区顺序以避免相邻社区颜色相近
node_community_shuffled = node_community.sample(frac=1, random_state=42)

# 绘制社区
node_community_shuffled.plot(
    cmap=ListedColormap(cmap),
    ax=ax,
    edgecolor='#333',
    linewidth=1.5,
    alpha=0.7
)

# 添加比例尺和指北针
tbd.plotscale(
    ax,
    bounds=BOUNDS,
    textsize=10,
    compasssize=1,
    textcolor='black',
    accuracy=2000,
    rect=[0.06, 0.03],
    zorder=10
)

plt.axis('off')
plt.xlim(BOUNDS[0], BOUNDS[2])
plt.ylim(BOUNDS[1], BOUNDS[3])

# 保存PNG
png_path = get_result_path(OUTPUT_SECTION, 'star_社区发现结果.png')
plt.savefig(png_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig)

print(f"社区发现PNG已保存: {png_path}")

# 保存模块度报告
report_path = get_result_path(OUTPUT_SECTION, '社区发现_模块度报告.txt')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(f"社区发现结果报告\n")
    f.write(f"{'='*40}\n")
    f.write(f"算法: Louvain (igraph.community_multilevel)\n")
    f.write(f"网格精度: 500m\n")
    f.write(f"节点数（网格）: {len(node)}\n")
    f.write(f"边数（OD对）: {len(edge)}\n")
    f.write(f"模块度 (Modularity): {modularity:.6f}\n")
    f.write(f"总社区数: {node['group'].nunique()}\n")
    f.write(f"大社区数（>10网格）: {len(large_groups)}\n")
    f.write(f"最大社区网格数: {group_counts.max()}\n")
    f.write(f"最小社区网格数: {group_counts.min()}\n")

print(f"模块度报告已保存: {report_path}")

## 3. 汇总

In [ ]:
print("\n" + "="*60)
print("动态行为层可视化完成！")
print("="*60)
print(f"\n输出文件:")
print(f"1. OD流线图HTML: {get_result_path(OUTPUT_SECTION, 'star_OD流线图_实际格局.html')}")
print(f"2. OD流线图PNG: {get_result_path(OUTPUT_SECTION, 'star_OD流线图_实际格局.png')}")
print(f"3. 社区发现PNG: {get_result_path(OUTPUT_SECTION, 'star_社区发现结果.png')}")
print(f"4. 社区发现CSV: {get_result_path(OUTPUT_SECTION, 'star_社区发现_最终结果.csv')}")
print(f"5. 模块度报告: {get_result_path(OUTPUT_SECTION, '社区发现_模块度报告.txt')}")
print(f"\n中间文件:")
print(f"- 网格化OD: {get_result_path(OUTPUT_SECTION, '社区发现_01_网格化OD.csv')}")
print(f"- 节点: {get_result_path(OUTPUT_SECTION, '社区发现_02_节点.csv')}")
print(f"- 边: {get_result_path(OUTPUT_SECTION, '社区发现_03_边.csv')}")
print(f"- 社区分配: {get_result_path(OUTPUT_SECTION, '社区发现_04_社区分配.csv')}")
print(f"\n模块度: {modularity:.6f}")
print(f"社区数: {len(large_groups)}")